# Beginning

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import evaluate
import warnings
import ujson
import json
import os
import sys
import time
import pandas as pd
import torch
from collections import Counter, defaultdict
from tqdm import tqdm
from itertools import chain
# Ignore all warnings
warnings.filterwarnings("ignore")

from data_module import NERDataset, NERDataModule, token_classification_collate_fn
from data_module import *
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
from functools import partial
from transformers.integrations import WandbCallback
from datasets import load_dataset
from data_utils import *
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# Load package to evaluate models
seqeval = evaluate.load("seqeval")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create dataset for /data/

In [3]:
tag2label = {
    "ncbi_disease": tag2label_ncbi,
    "bc5cdr": tag2label_bc5cdr,
    "bc5cdr_chemical": tag2label_bc5cdr_chemical,
    "bc5cdr_disease": tag2label_bc5cdr_disease,
    "ebm_pico": tag2label_pico,
    "pico_extraction": tag2label_pico,
}

dataset_name = 'bc5cdr'
# dataset_name = 'bc5cdr_chemical'
# dataset_name = 'bc5cdr_disease'
# dataset_name = 'ncbi_disease'
# dataset_name = 'medmentions_st21pv'
# dataset_name = 'ebm_pico'
# dataset_name = 'pico_extraction'
# dataset_name = "chemprot"
dataset = load_bigbio_dataset(dataset_name)
docs = dataset_to_documents(dataset)

if dataset_name in ["ebm_pico"] :
    val_split_ids = dataset['train']['document_id'][:187]
    df = dataset_to_df(dataset=dataset, val_split_ids=val_split_ids, dataset_name = dataset_name)
elif dataset_name in ["pico_extraction"] :
    test_split_ids = dataset['train']['document_id'][:40]
    val_split_ids = dataset['train']['document_id'][40:80]
    df = dataset_to_df(dataset=dataset, val_split_ids=val_split_ids, test_split_ids=test_split_ids, dataset_name = dataset_name)
else :
    df = dataset_to_df(dataset=dataset)

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
})

In [5]:
df[:5]

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id
0,10027919,"[[0, 16]]",22-oxacalcitriol,[Chemical],[MESH:C051883],validation,None,10027919.1
23,10027919,"[[28, 57]]",secondary hyperparathyroidism,[Disease],[MESH:D006962],validation,None,10027919.2
32,10027919,"[[75, 92]]",low bone turnover,[Disease],[MESH:D001851],validation,None,10027919.3
1,10027919,"[[106, 119]]",renal failure,[Disease],[MESH:D051437],validation,None,10027919.4
5,10027919,"[[133, 143]]",Calcitriol,[Chemical],[MESH:D002117],validation,None,10027919.5


In [185]:
# df['type'].apply(str).unique()

In [6]:
if dataset_name == 'ncbi_disease':
    for idx, row in df.iterrows():
        row['type'][0] = 'Disease'

elif dataset_name == 'bc5cdr_chemical':
    df = df[df['type'].apply(lambda x: x[0] != 'Disease')]
    
elif dataset_name == 'bc5cdr_disease':
    df = df[df['type'].apply(lambda x: x[0] != 'Chemical')]

elif dataset_name == 'ebm_pico':
    for idx, row in df.iterrows():
        if row['type'][0].startswith("Participant"):
            row['type'][0] = 'Participant'
        elif row['type'][0].startswith("Outcome"):
            row['type'][0] = 'Outcome'
        elif row['type'][0].startswith("Intervention"):
            row['type'][0] = 'Intervention'

df[:5]

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id
0,10027919,"[[0, 16]]",22-oxacalcitriol,[Chemical],[MESH:C051883],validation,None,10027919.1
23,10027919,"[[28, 57]]",secondary hyperparathyroidism,[Disease],[MESH:D006962],validation,None,10027919.2
32,10027919,"[[75, 92]]",low bone turnover,[Disease],[MESH:D001851],validation,None,10027919.3
1,10027919,"[[106, 119]]",renal failure,[Disease],[MESH:D051437],validation,None,10027919.4
5,10027919,"[[133, 143]]",Calcitriol,[Chemical],[MESH:D002117],validation,None,10027919.5


In [7]:
doc_id = list(docs.keys())[0]
from IPython.display import HTML, display
display(HTML(f"<div style='white-space: pre-wrap; word-wrap: break-word;'>{docs[doc_id]}</div>"))

In [8]:
# Apply the replacement
processed_data = process_dataset(docs, df, tag2label[dataset_name])

In [9]:
pmid = df.iloc[0]['document_id']
processed_data[pmid]

{'input': '22-oxacalcitriol suppresses secondary hyperparathyroidism without inducing low bone turnover in dogs with renal failure.\nBACKGROUND: Calcitriol therapy suppresses serum levels of parathyroid hormone (PTH) in patients with renal failure but has several drawbacks, including hypercalcemia and/or marked suppression of bone turnover, which may lead to adynamic bone disease. A new vitamin D analogue, 22-oxacalcitriol (OCT), has been shown to have promising characteristics. This study was undertaken to determine the effects of OCT on serum PTH levels and bone turnover in states of normal or impaired renal function. METHODS: Sixty dogs were either nephrectomized (Nx, N = 38) or sham-operated (Sham, N = 22). The animals received supplemental phosphate to enhance PTH secretion. Fourteen weeks after the start of phosphate supplementation, half of the Nx and Sham dogs received doses of OCT (three times per week); the other half were given vehicle for 60 weeks. Thereafter, the treatment

In [10]:
check_test_length = [pmid for pmid in processed_data.keys() if processed_data[pmid]['split']=='test']
check_valid_length = [pmid for pmid in processed_data.keys() if processed_data[pmid]['split']=='validation']
len(check_test_length), len(check_valid_length)

(500, 500)

In [12]:
with open(f"/home2/cye73/noisyNER/noisyNER/data/{dataset_name}/{dataset_name}.json", "w") as f:
    ujson.dump(processed_data, f, indent=2)

In [11]:
f"/home2/cye73/noisyNER/noisyNER/data/{dataset_name}/{dataset_name}.json"

'/home2/cye73/noisyNER/noisyNER/data/bc5cdr/bc5cdr.json'

# Create dataset for /data2/

In [13]:
def process_dataset_data2(docs, df, tag2label):
    """
    Function which modifies the abstract to include the entity:
    Input : DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection
    Output : @@DCTN4##T103@@ as a modifier of @@chronic Pseudomonas aeruginosa infection##T038@@
    -------------
    docs : dict {pmid : abstract}
    df : pd.DataFrame
    """
    processed_data = []
    for pmid in docs:
        # Perform replacements based on offsets
        spans = []
        new_df = df[df["document_id"] == pmid]
        for _, row in new_df.iterrows():
            if len(row["offsets"]) > 1:
                continue
            start, end = row["offsets"][0]
            text = row["text"]
            label = tag2label[row["type"][0]]  # label
            tag = row["type"][0]
            span = {
                "start": start,
                "end": end,
                "label": label,
                "tag": tag,
                "text": text,
            }
            spans.append(span)

        processed_data.append({
            "text": docs[pmid],
            "spans": spans,
            "pmid": pmid,
            "split": row["split"],
        })
    return processed_data

In [14]:
processed_data2 = process_dataset_data2(docs, df, tag2label[dataset_name])

In [15]:
processed_data2[0]

{'text': 'Naloxone reverses the antihypertensive effect of clonidine.\nIn unanesthetized, spontaneously hypertensive rats the decrease in blood pressure and heart rate produced by intravenous clonidine, 5 to 20 micrograms/kg, was inhibited or reversed by nalozone, 0.2 to 2 mg/kg. The hypotensive effect of 100 mg/kg alpha-methyldopa was also partially reversed by naloxone. Naloxone alone did not affect either blood pressure or heart rate. In brain membranes from spontaneously hypertensive rats clonidine, 10(-8) to 10(-5) M, did not influence stereoselective binding of [3H]-naloxone (8 nM), and naloxone, 10(-8) to 10(-4) M, did not influence clonidine-suppressible binding of [3H]-dihydroergocryptine (1 nM). These findings indicate that in spontaneously hypertensive rats the effects of central alpha-adrenoceptor stimulation involve activation of opiate receptors. As naloxone and clonidine do not appear to interact with the same receptor site, the observed functional antagonism suggests th

In [17]:
with open(f"/home2/cye73/noisyNER/noisyNER/data2/{dataset_name}/{dataset_name}.json", "w") as f:
    ujson.dump(processed_data2, f, indent=2)

In [16]:
f"/home2/cye73/noisyNER/noisyNER/data2/{dataset_name}/{dataset_name}.json"

'/home2/cye73/noisyNER/noisyNER/data2/bc5cdr/bc5cdr.json'

In [ ]:
# with open(f"/home2/cye73/noisyNER/noisyNER/data2/{dataset_name}/{dataset_name}.json", "r") as f:
#     processed_data_old = ujson.load(f)

In [62]:
for idx in range(len(processed_data2)):
    if processed_data2[idx] != processed_data_old[idx]:
        print(idx, processed_data2[idx]['pmid'])
    

8 2234245
14 2670794
20 3412544
27 3997294
80 19631624
81 20003049
84 20722491
87 322550
91 1835291
94 2054792
138 11587867
151 15458908
157 16904497
173 19957053
192 3503576
202 7053303
214 9284778
250 17854040
262 19729346
294 18821488
309 11532387
341 84204
342 9862868
349 2334618
354 230316
363 2224762
365 12483326
389 15863244
390 12921865
393 3403780
417 2375138
455 20528871
457 11928786
459 19300402
465 20477932
519 24100055
549 24618873
560 24709919
564 24733133
569 24802403
603 920167
617 19681452
618 20009434
662 1837756
665 1943082
669 2173761
680 2614930
765 7479194
766 7492040
812 9725303
823 10401555
837 10985896
843 11229942
847 11385188
854 11745287
873 12677626
879 12948256
914 16723784
933 17346443
937 17384765
983 20046642
1006 3115150
1009 2339463
1013 591536
1071 6415512
1079 18674790
1084 16428827
1094 6150641
1110 11185967
1147 16629641
1159 2322844
1166 17943461
1183 9758264
1220 8638876
1224 8643966
1225 6695415
1227 8800187
1234 20683499
1237 19721134
1238 177

In [64]:
processed_data_old[8]

{'text': 'Ocular and auditory toxicity in hemodialyzed patients receiving desferrioxamine.\nDuring an 18-month period of study 41 hemodialyzed patients receiving desferrioxamine (10-40 mg/kg BW/3 times weekly) for the first time were monitored for detection of audiovisual toxicity. 6 patients presented clinical symptoms of visual or auditory toxicity. Moreover, detailed ophthalmologic and audiologic studies disclosed abnormalities in 7 more asymptomatic patients. Visual toxicity was of retinal origin and was characterized by a tritan-type dyschromatopsy, sometimes associated with a loss of visual acuity and pigmentary retinal deposits. Auditory toxicity was characterized by a mid- to high-frequency neurosensorial hearing loss and the lesion was of the cochlear type. Desferrioxamine withdrawal resulted in a complete recovery of visual function in 1 patient and partial recovery in 3, and a complete reversal of hearing loss in 3 patients and partial recovery in 3. This toxicity appeared i

In [58]:
processed_data2[8]

{'text': 'Comparison of three solutions of ropivacaine/fentanyl for postoperative patient-controlled epidural analgesia . BACKGROUND Ropivacaine , 0.2 % , is a new local anesthetic approved for epidural analgesia . The addition of 4 microg/ml fentanyl improves analgesia from epidural ropivacaine . Use of a lower concentration of ropivacaine-fentanyl may further improve analgesia or decrease side effects . METHODS Thirty patients undergoing lower abdominal surgery were randomized in a double-blinded manner to receive one of three solutions : 0.2 % ropivacaine-4 microg fentanyl 0.1 % ropivacaine-2 microg fentanyl , or 0.05 % ropivacaine-1 microg fentanyl for patient-controlled epidural analgesia after standardized combined epidural and general anesthesia . Patient-controlled epidural analgesia settings and adjustments for the three solutions were standardized to deliver equivalent drug doses . Pain scores ( rest , cough , and ambulation ) , side effects ( nausea , pruritus , sedation , m

# Experiment

In [ ]:
dataset_name = "ebm_pico"
data_path = f"/home2/cye73/noisyNER/noisyNER/data2/{dataset_name}/{dataset_name}.json"
hf_model = "michiyasunaga/BioLinkBERT-base"
data_module = NERDataModule(
        data_path,
        hf_model=hf_model,
        max_length=1000,
        debug=False,
    )
data_module.setup()
train = data_module.train
validation = data_module.validation
test = data_module.test
tokenizer = data_module.tokenizer

In [ ]:
# for i in range(len(train)):
#     if len(train[i][1]) > 500:
#         print("index :", i)
#         print(len(train[i][1]))


In [21]:
data_module.split_data['train'][0]

{'text': 'DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis\nPseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with reduced lung function, faster rate of lung decline, increased rates of exacerbations and shorter survival. By using exome sequencing and extreme phenotype design, it was recently shown that isoforms of dynactin 4 (DCTN4) may influence Pa infection in CF, leading to worse respiratory disease. The purpose of this study was to investigate the role of DCTN4 missense variants on Pa infection incidence, age at first Pa infection and chronic Pa infection incidence in a cohort of adult CF patients from a single centre. Polymerase chain reaction and direct sequencing were used to screen DNA samples for DCTN4 variants. A total of 121 adult CF patients from the Cochin Hospital CF centre have been included, all o

In [20]:
train[453]

('Regulation of the host immune system by helminth parasites\nHelminth parasite infections are associated with a battery of immunomodulatory mechanisms, which impact all facets of the host immune response to ensure their persistence within the host. This broad-spectrum modulation of host immunity has intended and unintended consequences, both advantageous and disadvantageous. Thus the host may benefit from suppression of collateral damage during parasite infection, and from reduced allergic, autoimmune and inflammatory reactions. However, helminth infection can also be detrimental in reducing vaccine responses, increasing susceptibility to co-infection, and potentially reducing tumor immunosurveillance. In this review we will summarize the panoply of immunomodulatory mechanisms used by helminths, their potential utility in human disease, and prospective areas of future research.',
 tensor([ 0, 17,  0,  0,  0, 21, 22,  0, 27, 27, 17, 18, 18,  0,  0,  0,  0,  0,
          0, 17, 18,  0, 

In [21]:
id2label = data_module.id2label
label2id = data_module.label2id
num_labels = len(id2label)

In [20]:
collate_fn = partial(token_classification_collate_fn, tokenizer=tokenizer)

In [22]:
collate_fn

functools.partial(<function token_classification_collate_fn at 0x7f6eb66af560>, tokenizer=BertTokenizerFast(name_or_path='michiyasunaga/BioLinkBERT-base', vocab_size=28895, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [67]:
pmid = '10526274'
data_path = '/home2/cye73/noisyNER/noisyNER/data/bc5cdr.json'
data = ujson.load(open(data_path))
for element in data:
    if element.get('pmid') == pmid:
        target_element = element
        break
    
abstract = target_element
sent_span_idxs = get_sent_boundaries(sci_nlp, abstract['text'])
split_inds = get_abstract_split(abstract, tokenizer, max_len=500)
chunks = update_offsets(abstract, split_inds)

In [98]:
split_data = defaultdict(list)
span_tags = set([])
split_names = set([])

if len(split_inds) > 1:
    chunks = update_offsets(abstract, split_inds)

    # Make sure remapings were performed corectly
    for chunk in chunks:
        text = chunk["text"]
        for span in chunk["spans"]:
            span_tags.add(span["tag"])
            assert text[span["start"] : span["end"]] == span["text"]
else:
    for span in abstract["spans"]:
        span_tags.add(span["tag"])
    chunks = [abstract]

# logger.info('Getting token-level tags for each chunk')
for chunk in chunks:
    token_tags = get_token_labels_from_char_spans(tokenizer, chunk)
    chunk["token_tags"] = token_tags

split_data[abstract["split"]].extend(chunks)
split_names.add(abstract["split"])

In [99]:
split_data['train']

[{'chunk_id': 0,
  'pmid': '10526274',
  'text': 'Gemcitabine plus vinorelbine in nonsmall cell lung carcinoma patients age 70 years or older or patients who cannot receive cisplatin. Oncopaz Cooperative Group.\nBACKGROUND: Although the prevalence of nonsmall cell lung carcinoma (NSCLC) is high among elderly patients, few data are available regarding the efficacy and toxicity of chemotherapy in this group of patients. Recent reports indicate that single agent therapy with vinorelbine (VNB) or gemcitabine (GEM) may obtain a response rate of 20-30% in elderly patients, with acceptable toxicity and improvement in symptoms and quality of life. In the current study the efficacy and toxicity of the combination of GEM and VNB in elderly patients with advanced NSCLC or those with some contraindication to receiving cisplatin were assessed. METHODS: Forty-nine patients with advanced NSCLC were included, 38 of whom were age >/= 70 years and 11 were age < 70 years but who had some contraindication

In [87]:
len(chunks)

2

In [94]:
def get_token_labels_from_char_spans(tokenizer, span_data: Dict[str, any]) -> List[str]:
    # Tokenize the input text and get the offset mapping
    tokenized = tokenizer(span_data["text"], return_offsets_mapping=True)

    # Create an empty list of tags, one for each token
    tags = ["O"] * len(tokenized["input_ids"])

    # Go through each span in the data
    for span in span_data["spans"]:
        span_started = False
        # Go through each offset mapping
        for i, (start, end) in enumerate(tokenized["offset_mapping"]):
            # Skip special tokens
            if start == end == 0:
                continue

            # If the start of the token is inside the span, start the span
            if start >= span["start"] and start < span["end"]:
                if not span_started:
                    tags[i] = f"B-{span['tag']}"
                    span_started = True
                else:
                    tags[i] = f"I-{span['tag']}"

            # If the token is inside the span, continue the span
            elif start < span["start"] and end > span["start"]:
                tags[i] = f"B-{span['tag']}"
                span_started = True

    return tags

In [30]:
from typing import List, Dict, Union
from transformers import PreTrainedTokenizer, DataCollatorForTokenClassification
from torch.utils.data import Dataset
import torch

class DatasetNER(Dataset):
    def __init__(self, data: List[Dict], tokenizer: PreTrainedTokenizer, label2id: Dict[str, int], max_length: int = 512, split: str = "train"):
        """
        Args:
            data (List[Dict]): The input dataset in the provided format.
            tokenizer (PreTrainedTokenizer): Tokenizer for the model.
            label2id (Dict[str, int]): Mapping of labels to integer IDs.
            max_length (int): Maximum sequence length.
            split (str): The split of the dataset ("train", "valid", "test").
        """
        self.data = [entry for entry in data if entry["split"] == split]
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
        self.inputs = []
        self.labels = []

        self._prepare_dataset()

    def _prepare_dataset(self):
        for entry in self.data:
            text = entry["text"]
            spans = entry.get("spans", [])

            # Tokenize the text
            tokenized = self.tokenizer(text, truncation=True, max_length=self.max_length, return_offsets_mapping=True)
            input_ids = tokenized["input_ids"]
            attention_mask = tokenized["attention_mask"]
            offset_mapping = tokenized["offset_mapping"]

            # Initialize labels with 'O' for all tokens
            labels = ["O"] * len(input_ids)

            # Assign labels to tokens based on spans
            for span in spans:
                span_start, span_end, tag = span["start"], span["end"], span["tag"]

                # BIO tagging (B-label for beginning, I-label for inside)
                for idx, (start, end) in enumerate(offset_mapping):
                    if start >= span_start and end <= span_end:
                        if labels[idx] == "O":  # Assign only if not labeled yet
                            if start == span_start:
                                labels[idx] = f"B-{tag}"  # Beginning of the entity
                            else:
                                labels[idx] = f"I-{tag}"  # Inside of the entity

            # Convert labels to numerical IDs
            label_ids = [self.label2id[label] if label in self.label2id else self.label2id["O"] for label in labels]

            self.inputs.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask
            })
            self.labels.append(label_ids)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        item = self.inputs[idx]
        item["labels"] = self.labels[idx]
        return {key: torch.tensor(val) for key, val in item.items()}

def create_label2id(data: List[Dict]) -> Dict[str, int]:
    """
    Automatically create label2id mapping from dataset.

    Args:
        data (List[Dict]): The input dataset in the provided format.

    Returns:
        Dict[str, int]: Mapping of unique labels to numerical IDs.
    """
    unique_labels = {"O"}  # Start with "O" for outside tokens
    for entry in data:
        spans = entry.get("spans", [])
        for span in spans:
            tag = span["tag"]
            unique_labels.add(f"B-{tag}")  # Beginning tag
            unique_labels.add(f"I-{tag}")  # Inside tag

    # Create label2id dictionary sorted alphabetically for consistent order
    return {label: idx for idx, label in enumerate(sorted(unique_labels))}

# Usage
data_path = '/home2/cye73/noisyNER/noisyNER/data/bc5cdr.json'
data = ujson.load(open(data_path))

# Example tokenizer and label mapping
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
label2id = create_label2id(data)

# Create datasets for each split
train_dataset = DatasetNER(data, tokenizer, label2id, split="train")
valid_dataset = DatasetNER(data, tokenizer, label2id, split="valid")
test_dataset = DatasetNER(data, tokenizer, label2id, split="test")

# Example collator
collator = DataCollatorForTokenClassification(tokenizer)

# Collate batch example for train
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=1, collate_fn=collator)

# Example batch
for batch in train_dataloader:
    print(batch)
    break


{'input_ids': tensor([[  101, 11896,  2858, 21501,  1162,  7936,  1116,  1103,  2848,  7889,
         17786,  5026,  2109,  2629,  1104,   172,  4934,  2386,  2042,   119,
          1130,  8362,  6354,  2050,  4638, 26300,   117, 20061,  1193,   177,
         24312, 27291, 13475,  1103,  9711,  1107,  1892,  2997,  1105,  1762,
          2603,  1666,  1118,  1107,  4487,  7912,  2285,   172,  4934,  2386,
          2042,   117,   126,  1106,  1406, 17599, 12139,  1116,   120,  4023,
           117,  1108,  1107, 23034,  1174,  1137, 11802,  1118,  9468,  2858,
         19315,   117,   121,   119,   123,  1106,   123, 17713,   120,  4023,
           119,  1109,   177,  1183, 11439,  5026,  2109,  2629,  1104,  1620,
         17713,   120,  4023, 11164,   118,  1899, 18873,  2572,  4163,  1108,
          1145,  6320, 11802,  1118,  9468,  2858, 21501,  1162,   119, 11896,
          2858, 21501,  1162,  2041,  1225,  1136,  6975,  1719,  1892,  2997,
          1137,  1762,  2603,   119,  

In [31]:
label2id

{'B-Chemical': 0, 'B-Disease': 1, 'I-Chemical': 2, 'I-Disease': 3, 'O': 4}

In [29]:
train_dataset[0]

{'input_ids': tensor([  101, 11896,  2858, 21501,  1162,  7936,  1116,  1103,  2848,  7889,
         17786,  5026,  2109,  2629,  1104,   172,  4934,  2386,  2042,   119,
          1130,  8362,  6354,  2050,  4638, 26300,   117, 20061,  1193,   177,
         24312, 27291, 13475,  1103,  9711,  1107,  1892,  2997,  1105,  1762,
          2603,  1666,  1118,  1107,  4487,  7912,  2285,   172,  4934,  2386,
          2042,   117,   126,  1106,  1406, 17599, 12139,  1116,   120,  4023,
           117,  1108,  1107, 23034,  1174,  1137, 11802,  1118,  9468,  2858,
         19315,   117,   121,   119,   123,  1106,   123, 17713,   120,  4023,
           119,  1109,   177,  1183, 11439,  5026,  2109,  2629,  1104,  1620,
         17713,   120,  4023, 11164,   118,  1899, 18873,  2572,  4163,  1108,
          1145,  6320, 11802,  1118,  9468,  2858, 21501,  1162,   119, 11896,
          2858, 21501,  1162,  2041,  1225,  1136,  6975,  1719,  1892,  2997,
          1137,  1762,  2603,   119,  1

In [23]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class DatasetNER(Dataset):
    def __init__(self, data, tokenizer, label2id, split='train', max_length=512):
        """
        Initializes the DatasetNER class.

        Args:
            data (list): List of dictionaries containing 'text' and 'spans'.
            tokenizer (transformers.PreTrainedTokenizer): Tokenizer to use.
            label2id (dict): Mapping from label strings to integer IDs.
            split (str): Dataset split to use ('train', 'validation', 'test').
            max_length (int): Maximum sequence length.
        """
        self.data = [item for item in data if item['split'] == split]
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
        self.examples = self.process_data()
        
    def process_data(self):
        """
        Processes the data by tokenizing text and aligning labels.

        Returns:
            list: A list of tokenized inputs with aligned labels.
        """
        examples = []
        for item in self.data:
            text = item['text']
            spans = item['spans']
            entities = []
            for span in spans:
                entities.append({
                    'start': span['start'],
                    'end': span['end'],
                    'label': span['tag']
                })
            tokenized_input = self.tokenizer(
                text,
                truncation=True,
                max_length=self.max_length,
                return_offsets_mapping=True,
                padding='max_length'
            )
            labels = self.align_labels(tokenized_input['offset_mapping'], entities)
            tokenized_input["labels"] = labels
            # Remove offset_mapping as it's no longer needed
            tokenized_input.pop("offset_mapping")
            examples.append(tokenized_input)
        return examples

    def align_labels(self, offset_mapping, entities):
        """
        Aligns labels with tokens based on offset mappings.

        Args:
            offset_mapping (list): List of (start, end) tuples for each token.
            entities (list): List of entities with 'start', 'end', and 'label'.

        Returns:
            list: A list of label IDs aligned with the tokens.
        """
        labels = ['O'] * len(offset_mapping)
        for idx, (start, end) in enumerate(offset_mapping):
            if start == end:
                # Special token like [CLS], [SEP], or padding
                labels[idx] = -100
                continue
            for entity in entities:
                entity_start = entity['start']
                entity_end = entity['end']
                entity_label = entity['label']
                # Check if the token is within the entity span
                if start >= entity_start and end <= entity_end:
                    if start == entity_start:
                        labels[idx] = 'B-' + entity_label
                    else:
                        labels[idx] = 'I-' + entity_label
                    break  # Break if the token has been labeled
        # Convert labels to IDs
        labels = [self.label2id[label] if label != -100 else -100 for label in labels]
        return labels

    def __getitem__(self, idx):
        """
        Retrieves an item by index.

        Args:
            idx (int): Index of the item.

        Returns:
            dict: A dictionary containing tokenized inputs and labels.
        """
        return self.examples[idx]

    def __len__(self):
        """
        Returns the total number of examples.

        Returns:
            int: Number of examples.
        """
        return len(self.examples)


In [ ]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification

# Sample data (replace this with your actual data)
data_path = '/home2/cye73/noisyNER/noisyNER/data2/bc5cdr.json'
data = ujson.load(open(data_path))

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Define label mappings
label_list = ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']  # Add other labels as needed
label2id = {label: idx for idx, label in enumerate(label_list)}

# Create datasets for train, validation, and test
train_dataset = DatasetNER(data, tokenizer, label2id, split='train')
validation_dataset = DatasetNER(data, tokenizer, label2id, split='validation')
test_dataset = DatasetNER(data, tokenizer, label2id, split='test')

# Initialize data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Example of creating DataLoaders
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=2, collate_fn=data_collator)
validation_dataloader = DataLoader(validation_dataset, batch_size=2, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=2, collate_fn=data_collator)

train_dataloader = DataLoader(train_dataset, batch_size=1, collate_fn=data_collator)

# Example batch
for batch in train_dataloader:
    print(batch)
    break


{'input_ids': tensor([[  101,  6583,  4135, 22500,  2063,  7901,  2015,  1996,  3424, 10536,
          4842, 25808,  3512,  3466,  1997, 18856, 10698, 10672,  1012,  1999,
         14477,  5267, 10760, 23355,  1010, 27491, 23760, 25808,  3512, 11432,
          1996,  9885,  1999,  2668,  3778,  1998,  2540,  3446,  2550,  2011,
         26721,  8159,  3560, 18856, 10698, 10672,  1010,  1019,  2000,  2322,
         12702, 13113,  2015,  1013,  4705,  1010,  2001, 26402,  2098,  2030,
         11674,  2011,  6583,  4135, 15975,  1010,  1014,  1012,  1016,  2000,
          1016, 11460,  1013,  4705,  1012,  1996,  1044, 22571, 12184,  3619,
          3512,  3466,  1997,  2531, 11460,  1013,  4705,  6541,  1011, 25003,
          3527,  4502,  2001,  2036,  6822, 11674,  2011,  6583,  4135, 22500,
          2063,  1012,  6583,  4135, 22500,  2063,  2894,  2106,  2025,  7461,
          2593,  2668,  3778,  2030,  2540,  3446,  1012,  1999,  4167, 24972,
          2013, 27491, 23760, 25808,  